In [ ]:
module load anaconda3/2022.05-gcc/9.5.0 cuda/11.1.1-gcc/9.5.0 cudnn/8.0.5.39-11.1-gcc/9.5.0-cu11_1

source activate tf_v2.2_env

In [ ]:
tf.config.list_physical_devices('GPU')

In [1]:
import logging
logging.getLogger('tensorflow').disabled = True

# TensorFlow, keras, np
import tensorflow as tf
from tensorflow import keras
import math
import numpy as np
import time
import sys

from io import UnsupportedOperation
import pickle as pkl

# Add shared location for auxillary functions
sys.path.insert(1, './../AuxillaryFunctions')


# import personal Functions
from GenerateClassDataFuncs import GenerateDataAndClasses_ClemCafe_Bite
from GenerateClassDataFuncs import GenerateDataAndClasses_OREBA_Bite


print("Finished importing libraries and functions.")

Finished importing libraries and functions.


In [2]:
############################
##### DEFINE CONSTANTS #####
############################

CUT = 128 # 2 seconds @ 64 Hz 

MODEL_NAME = 'Example_HeydCNN_Clemson/models/fold3'

DATABASE_FLAG = 3 # Must be 1 (Dom Hand Only) or 2 (Both Hands) or 3(ClemCafe Data)

RR_FLAG = 1 # 0 = Block Fold Segmentation, 1 = Striped/Round Robin segmentation

FOLD_INDEX = 3 # Which index to leave out for validation
FOLDS_TOTAL = 5 # total number of folds to split the data into

# 	# DOWN SAMPLE DATA INPUTS
DOWN_SAMPLE_FLAG = 0 # 1 = Down Sample data, 0 = use raw data
DOWN_SAMPLE_RATE = 1 # Num of points to consolidate into a single point
DOWN_SAMPLE_OFFSET = 0 # Offset in DS data; in range [0, DS_Rate) 

# NUMBER OF HANDS
if DATABASE_FLAG == 1:
	DATABASE_FILEPATH = './../Pickle_Databases/OREBA.pkl'
	total_axes=6
	STRIDE = 32 # number of points to jump @ 64 Hz
elif DATABASE_FLAG == 2:
	DATABASE_FILEPATH = './../Pickle_Databases/OREBA_TwoHands.pkl'
	total_axes=12
	STRIDE = 32 # number of points to jump @ 64 Hz
elif DATABASE_FLAG == 3:
	DATABASE_FILEPATH = './../Pickle_Databases/ClemCafe.pkl'
	total_axes=6
	STRIDE = 96 # number of points to jump @ 15 Hz; Jumping forward by 3 seconds each to limit training examples due to large size of data
else:
	print("INVALID DATABASE SELECTION FLAG VALUE OF {}".format(DATABASE_FLAG))
	exit(0)
# end of Database_Flag

print("Finished Defining Constants.")


Finished Defining Constants.


In [3]:
#########################
##### KNOBS TO TURN #####
#########################

### Knobs to Turn ###
NUM_EPOCHS = 8 # Appears to stabilize after 3, so dropping from 30 to 5 since model takes a while to train
WINDOWS_PER_BATCH = 256 # the number of samples to pull from files each time 
				# Value should maintain inequality [STRIDE*WINDOWS_PER_BATCH < 4.5M * (#folds-1)/#folds]
USING_CLASS_WEIGHTS_FLAG=0
if USING_CLASS_WEIGHTS_FLAG==1:
	# Weighting to be used for classes; 1 is undersampled and should have a higher weight
	Class_Weights_OREBA = {0: 1.0, 1: 4.0}

# Define Variables 
sample_length = CUT
num_samples=WINDOWS_PER_BATCH

print('CUT Input is ',CUT)
print('STRIDE input is ',STRIDE)
print('Training Data coming from file ',DATABASE_FILEPATH)
print('FOLD_INDEX input is ',FOLD_INDEX)
print('FOLDS_TOTAL input is ',FOLDS_TOTAL)

print("\nPython setup complete")


CUT Input is  128
STRIDE input is  96
Training Data coming from file  ./../Pickle_Databases/ClemCafe.pkl
FOLD_INDEX input is  3
FOLDS_TOTAL input is  5

Python setup complete


In [4]:
#######################################
##### GENERATE MODEL ARCHITECTURE #####
#######################################

model = keras.Sequential()
### FIRST PHASE: MICROMOVEMENT GESTURE PROBABILITIES ###

#input as Convolution
model.add(keras.layers.Conv1D(input_shape=(sample_length,total_axes,),
		filters=128,kernel_size=1,
		#strides=15,
		padding='valid',
		activation='relu'))

# Additional Convolution Layers
model.add(keras.layers.Conv1D(filters=128,kernel_size=3,
		padding='valid',
		activation='relu'))
model.add(keras.layers.Conv1D(filters=128,kernel_size=5,
		padding='valid',
		activation='relu'))
model.add(keras.layers.Conv1D(filters=128,kernel_size=7,
		padding='valid',
		activation='relu'))


### SECOND PHASE: TIME MEMORY OF GESTURES ###
model.add(keras.layers.LSTM(64, return_sequences=True,
		activation="tanh",recurrent_activation="hard_sigmoid"))

model.add(keras.layers.LSTM(64, 
		activation="tanh",recurrent_activation="hard_sigmoid"))


model.add(keras.layers.Flatten())  # must flatten to feed dense layer
model.add(keras.layers.Dense(1))



model.compile(optimizer='adam',
			  loss='mean_squared_error',
			  metrics=['mean_absolute_error'])
#model.compile(optimizer='adam',
#			  loss='binary_crossentropy',
#			  metrics=['binary_crossentropy'])

model.summary()


Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv1d (Conv1D)              (None, 128, 128)          896       
_________________________________________________________________
conv1d_1 (Conv1D)            (None, 126, 128)          49280     
_________________________________________________________________
conv1d_2 (Conv1D)            (None, 122, 128)          82048     
_________________________________________________________________
conv1d_3 (Conv1D)            (None, 116, 128)          114816    
_________________________________________________________________
lstm (LSTM)                  (None, 116, 64)           49408     
_________________________________________________________________
lstm_1 (LSTM)                (None, 64)                33024     
_________________________________________________________________
flatten (Flatten)            (None, 64)                0

In [5]:
#######################################
##### GENERATE MODEL ARCHITECTURE #####
#######################################


# Create Training Data and Classes
if DATABASE_FLAG == 3: # If Using ClemCafe Data at 15 Hz
	training_data,classes=GenerateDataAndClasses_ClemCafe_Bite(
		CUT, STRIDE, DATABASE_FILEPATH, 
		FOLD_INDEX, FOLDS_TOTAL, FOLD_SPLIT=RR_FLAG, RESAMPLE_FLAG=64
	)
else: # if using OREBA data at 64 Hz
	training_data,classes=GenerateDataAndClasses_OREBA_Bite(
		CUT, STRIDE, DATABASE_FILEPATH, 
		FOLD_INDEX, FOLDS_TOTAL, 
		FOLD_SPLIT = RR_FLAG, 
		DS_FLAG = DOWN_SAMPLE_FLAG,
		DS_Rate = DOWN_SAMPLE_RATE,
		DS_Delay = DOWN_SAMPLE_OFFSET
	)

print("training_data of size ",training_data.shape)
print("Classes of size ",classes.shape)
print("Total Positive Class Examples: ",sum(classes))


print("\nFinished Generating Training Classes and Targets.")


Training on Data Split by Round Robin approach...
training_data of size  (197783, 128, 6)
Classes of size  (197783,)
Total Positive Class Examples:  35446

Finished Generating Training Classes and Targets.


In [ ]:
#######################
##### TRAIN MODEL #####
#######################


print("Training")
start=time.time()

if USING_CLASS_WEIGHTS_FLAG==0:
	metrics = model.fit(training_data, classes, epochs=NUM_EPOCHS,
				validation_split=0.15, verbose=2)
elif USING_CLASS_WEIGHTS_FLAG==1:
	print("Using Class Weights for Training...")
	print(Class_Weights_OREBA)
	metrics = model.fit(training_data, classes, epochs=NUM_EPOCHS,
						class_weight=Class_Weights_OREBA,					
						validation_split=0.15, verbose=2)
# end of if class weights

end=time.time()
print("classifier training",end-start," seconds")

Training
Epoch 1/8
5254/5254 - 3729s - loss: 0.1062 - mean_absolute_error: 0.2109 - val_loss: 0.1017 - val_mean_absolute_error: 0.1950
Epoch 2/8
5254/5254 - 3430s - loss: 0.0966 - mean_absolute_error: 0.1925 - val_loss: 0.1069 - val_mean_absolute_error: 0.2025
Epoch 3/8
5254/5254 - 3320s - loss: 0.0937 - mean_absolute_error: 0.1869 - val_loss: 0.1007 - val_mean_absolute_error: 0.1930
Epoch 4/8
5254/5254 - 3321s - loss: 0.0901 - mean_absolute_error: 0.1800 - val_loss: 0.1008 - val_mean_absolute_error: 0.1872
Epoch 5/8
5254/5254 - 3514s - loss: 0.0877 - mean_absolute_error: 0.1756 - val_loss: 0.1018 - val_mean_absolute_error: 0.1953
Epoch 6/8


In [ ]:
#######################################
##### SAVE MODEL AND FINAL CHECKS #####
#######################################

# save model
#model.save((sys.argv[1]+'fold'+sys.argv[2]+'of'+sys.argv[3]+'.h5'))
#save model with lots of variable data in name

#model.save((sys.argv[1]+'.h5')) #concise model name
model.save(MODEL_NAME) #concise model name

print("Testing")


max_num_samples = min(40, len(training_data))
data_sample=training_data[0:max_num_samples]
classes_sample=classes[0:max_num_samples]

test_loss, test_acc = model.evaluate(data_sample, classes_sample)
print('Test accuracy:', test_acc)



print("Data Has input of shape:")
print(data_sample.shape)

predictions = model.predict(data_sample)
print('Actual, prediction:')
#for a in range(0,len(classes)):
for a in range(0,max_num_samples):
	print(classes_sample[a],predictions[a])
# end of for samples

print("\nFinished Running Training Script!")
    
    


In [ ]:
print("ALL DONE!")